In [1]:
#setup
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

BASE = Path(r"C:\Users\Hp\Downloads\Project 2026 DS")

LONDON_BOROUGHS = [
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley", "Camden",
    "City of London", "Croydon", "Ealing", "Enfield", "Greenwich", "Hackney",
    "Hammersmith and Fulham", "Haringey", "Harrow", "Havering", "Hillingdon",
    "Hounslow", "Islington", "Kensington and Chelsea", "Kingston upon Thames",
    "Lambeth", "Lewisham", "Merton", "Newham", "Redbridge",
    "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
    "Waltham Forest", "Wandsworth", "Westminster"
]

print("Config loaded. Base folder:", BASE)

Config loaded. Base folder: C:\Users\Hp\Downloads\Project 2026 DS


In [2]:
#boundary file
all_las = gpd.read_file(BASE / "boundaryfile.geojson")
boroughs = all_las[all_las["LAD24NM"].isin(LONDON_BOROUGHS)].copy()

print(f"Found {len(boroughs)} / 33 London boroughs")
missing_boroughs = set(LONDON_BOROUGHS) - set(boroughs["LAD24NM"])
if missing_boroughs:
    print("  WARNING — not matched in boundary file:", missing_boroughs)

Found 33 / 33 London boroughs


In [24]:
#functions

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def join_parent_child(child_items, parent_items, link_field, parent_id_field="@id"):
    parent_lookup = {}
    for item in parent_items:
        d = item.get("data", {})
        loc = d.get("location", {}) or {}
        geo = loc.get("geo", {}) or {}

        activity_list = d.get("activity", [{}]) or [{}]
        activity_value = activity_list[0].get("prefLabel") if activity_list else None
        if not activity_value:
            activity_value = d.get("name")  # fallback: use the session/series name itself

        parent_lookup[d.get(parent_id_field)] = {
            "venue_name": loc.get("name") or d.get("name"),
            "lat": geo.get("latitude"),
            "lon": geo.get("longitude"),
            "activity": activity_value,
        }

    rows = []
    for item in child_items:
        d = item.get("data", {})
        parent = parent_lookup.get(d.get(link_field), {})
        rows.append({
            "item_id": item.get("id"),
            "venue_name": parent.get("venue_name"),
            "lat": parent.get("lat"),
            "lon": parent.get("lon"),
            "activity": parent.get("activity"),
            "start_date": d.get("startDate"),
        })

    df = pd.DataFrame(rows)
    matched = df["lat"].notna().sum()
    print(f"  joined {len(df)} rows, {matched} matched to a venue ({matched/len(df)*100:.1f}%)")
    return df

def flatten_standalone(items, id_field="@id"):
    rows = []
    for item in items:
        d = item.get("data", {})
        loc = d.get("location", {}) or {}
        geo = loc.get("geo", {}) or {}

        activity_list = d.get("activity") or (loc.get("activity") or [{}])
        if not isinstance(activity_list, list):
            activity_list = [activity_list]
        activity_value = activity_list[0].get("prefLabel") if activity_list and isinstance(activity_list[0], dict) else None
        if not activity_value:
            activity_value = d.get("name")  # same fallback for standalone feeds

        rows.append({
            "item_id": d.get(id_field) or item.get("id"),
            "venue_name": loc.get("name") or d.get("name"),
            "lat": geo.get("latitude"),
            "lon": geo.get("longitude"),
            "activity": activity_value,
            "start_date": d.get("startDate"),
        })

    df = pd.DataFrame(rows)
    matched = df["lat"].notna().sum()
    print(f"  flattened {len(df)} rows, {matched} have coordinates ({matched/len(df)*100:.1f}%)")
    return df

def filter_to_london(df, boroughs_gdf):
    df = df.dropna(subset=["lat", "lon"]).copy()
    gdf = gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df.lon, df.lat)], crs="EPSG:4326")
    gdf = gdf.to_crs(boroughs_gdf.crs)
    joined = gpd.sjoin(gdf, boroughs_gdf[["LAD24NM", "geometry"]], how="left", predicate="within")
    london_only = joined[joined["index_right"].notna()].copy()
    london_only = london_only.drop(columns=["geometry", "index_right"])
    print(f"  {len(london_only)} / {len(df)} rows fall within a London borough")
    return london_only

print("Functions ready.")

Functions ready.


In [25]:
# join Session and Slot feeds
#places leisure

pl_sessions   = load_json(BASE / "Places_Leisure" / "ScheduledSession.json")
pl_series     = load_json(BASE / "Places_Leisure" / "SessionSeries.json")
pl_slots      = load_json(BASE / "Places_Leisure" / "Slot.json")
pl_facilities = load_json(BASE / "Places_Leisure" / "FacilityUse.json")

print("Places Leisure — Session:")
df_pl_session = join_parent_child(pl_sessions, pl_series, link_field="superEvent")
df_pl_session["provider"] = "Places Leisure"
df_pl_session["feed_type"] = "Session"

print("Places Leisure — Slot:")
df_pl_slot = join_parent_child(pl_slots, pl_facilities, link_field="facilityUse")
df_pl_slot["provider"] = "Places Leisure"
df_pl_slot["feed_type"] = "Slot"



Places Leisure — Session:
  joined 50164 rows, 44236 matched to a venue (88.2%)
Places Leisure — Slot:
  joined 154238 rows, 139478 matched to a venue (90.4%)


In [27]:
#join Session, flatten FacilityUse and CourseInstance
#everyone active

ea_sessions   = load_json(BASE / "Everyone_Active" / "ScheduledSession.json")
ea_series     = load_json(BASE / "Everyone_Active" / "SessionSeries.json")
ea_facilities = load_json(BASE / "Everyone_Active" / "FacilityUse.json")
ea_courses    = load_json(BASE / "Everyone_Active" / "CourseInstance.json")

print("Everyone Active — Session:")
df_ea_session = join_parent_child(ea_sessions, ea_series, link_field="superEvent")
df_ea_session["provider"] = "Everyone Active"
df_ea_session["feed_type"] = "Session"

print("Everyone Active — FacilityUse (standalone venue list):")
df_ea_facilities = flatten_standalone(ea_facilities)
df_ea_facilities["provider"] = "Everyone Active"
df_ea_facilities["feed_type"] = "FacilityUse"

print("Everyone Active — CourseInstance:")
df_ea_courses = flatten_standalone(ea_courses)
df_ea_courses["provider"] = "Everyone Active"
df_ea_courses["feed_type"] = "CourseInstance"


Everyone Active — Session:
  joined 135786 rows, 123567 matched to a venue (91.0%)
Everyone Active — FacilityUse (standalone venue list):
  flattened 2763 rows, 2612 have coordinates (94.5%)
Everyone Active — CourseInstance:
  flattened 26514 rows, 24713 have coordinates (93.2%)


In [6]:
#secro
serco_facilities = load_json(BASE / "Serco_Leisure" / "FacilityUse.json")

print("Serco Leisure — FacilityUse (standalone venue list):")
df_serco_facilities = flatten_standalone(serco_facilities)
df_serco_facilities["provider"] = "Serco Leisure"
df_serco_facilities["feed_type"] = "FacilityUse"

Serco Leisure — FacilityUse (standalone venue list):
  flattened 132 rows, 99 have coordinates (75.0%)


In [28]:
#london filter

all_frames = {
    "Places Leisure / Session": df_pl_session,
    "Places Leisure / Slot": df_pl_slot,
    "Everyone Active / Session": df_ea_session,
    "Everyone Active / FacilityUse": df_ea_facilities,
    "Everyone Active / CourseInstance": df_ea_courses,
    "Serco Leisure / FacilityUse": df_serco_facilities,
}

london_frames = []
for label, df in all_frames.items():
    print(f"{label}:")
    df_london = filter_to_london(df, boroughs)
    london_frames.append(df_london)

Places Leisure / Session:
  4361 / 44236 rows fall within a London borough
Places Leisure / Slot:
  12415 / 139478 rows fall within a London borough
Everyone Active / Session:
  20287 / 123567 rows fall within a London borough
Everyone Active / FacilityUse:
  477 / 2612 rows fall within a London borough
Everyone Active / CourseInstance:
  5007 / 24713 rows fall within a London borough
Serco Leisure / FacilityUse:
  0 / 99 rows fall within a London borough


In [29]:
# combine all data from providers

combined = pd.concat(london_frames, ignore_index=True)

print("Shape:", combined.shape)
print("\nBy provider:\n", combined["provider"].value_counts())
print("\nBy feed type:\n", combined["feed_type"].value_counts())
print("\nBy borough:\n", combined["LAD24NM"].value_counts())

combined.to_csv(BASE / "londonVenue_data.csv", index=False)
print("\nSaved londonvenue_data.csv")

# Sanity check on the fix — venue_name should no longer be blank for Slot rows
slot_check = combined[combined["feed_type"] == "Slot"]
print(f"\nSlot rows with a venue_name: {slot_check['venue_name'].notna().sum()} / {len(slot_check)}")

Shape: (42547, 9)

By provider:
 provider
Everyone Active    25771
Places Leisure     16776
Name: count, dtype: int64

By feed type:
 feed_type
Session           24648
Slot              12415
CourseInstance     5007
FacilityUse         477
Name: count, dtype: int64

By borough:
 LAD24NM
Wandsworth                12831
Havering                   5665
Kingston upon Thames       3945
Westminster                3719
Ealing                     3424
Sutton                     2752
Newham                     2568
Barking and Dagenham       2522
Brent                      2178
Harrow                     1543
Kensington and Chelsea      844
Barnet                      556
Name: count, dtype: int64

Saved londonvenue_data.csv

Slot rows with a venue_name: 12415 / 12415


In [30]:
# aggregate(vemue and activity)

venue_activity = (
    combined
    .groupby(["venue_name", "activity", "LAD24NM", "provider"], dropna=False)
    .agg(
        n_occurrences=("item_id", "count"),
        lat=("lat", "first"),
        lon=("lon", "first"),
    )
    .reset_index()
    .sort_values(["LAD24NM", "venue_name"])
)

print("Unique venue and activity rows:", len(venue_activity))
venue_activity.to_csv(BASE / "londonVenue_Activity.csv", index=False)
print("VenueActivity_London.csv saved")

Unique venue and activity rows: 14845
VenueActivity_London.csv saved


In [31]:
print(df_serco_facilities[["venue_name", "lat", "lon"]].drop_duplicates())
#all outside london

                                            venue_name        lat       lon
0                  Aqua Vale Swimming & Fitness Centre  51.816271 -0.801569
1            Bangor Aurora Aquatic and Leisure Complex  54.652403 -5.664621
8                                 Beacon Sports Centre        NaN       NaN
10                      Billesley Indoor Tennis Centre  52.428716 -1.876414
17                 Bisham Abbey National Sports Centre  51.556953 -0.777817
18                                SpArC Bishops Castle        NaN       NaN
21                            Bletchley Leisure Centre        NaN       NaN
34                    Cocks Moors Woods Leisure Centre  52.417650 -1.888312
39                               Evreham Sports Centre        NaN       NaN
41                                Ferry Leisure Centre        NaN       NaN
45                          Fox Hollies Leisure Centre  52.438914 -1.831622
49   Lilleshall National Sports and Conferencing Ce...  52.726849 -2.374770
52          

In [33]:
print(combined.groupby(["provider", "feed_type"])["lat"].apply(lambda x: x.isna().sum()))

provider         feed_type     
Everyone Active  CourseInstance    0
                 FacilityUse       0
                 Session           0
Places Leisure   Session           0
                 Slot              0
Name: lat, dtype: int64


In [34]:
print(combined["activity"].isna().sum(), "/", len(combined), "rows missing an activity label")

0 / 42547 rows missing an activity label


In [35]:
print(combined["item_id"].duplicated().sum())

12


In [14]:
dupes = combined[combined.duplicated(subset=["item_id"], keep=False)]
print(dupes[["item_id", "provider", "feed_type"]].sort_values("item_id"))

                              item_id         provider feed_type
164              00000000000130036917   Places Leisure   Session
4359             00000000000130036917   Places Leisure   Session
165              00000000000130036918   Places Leisure   Session
4360             00000000000130036918   Places Leisure   Session
163              00000000000140096770   Places Leisure   Session
4358             00000000000140096770   Places Leisure   Session
428              00000000000240048230   Places Leisure   Session
4058             00000000000240048230   Places Leisure   Session
432              00000000000240049270   Places Leisure   Session
3974             00000000000240049270   Places Leisure   Session
35836            00000000000345191164  Everyone Active   Session
25045            00000000000345191164  Everyone Active   Session
35971            00000000000345220630  Everyone Active   Session
16792            00000000000345220630  Everyone Active   Session
16822            00000000

In [36]:
print(sorted(venue_activity["activity"].dropna().unique()))

['1-2-1 Add Needs Mon 16:00', '1-2-1 Add Needs Mon 16:30', '1-2-1 Add Needs Mon 17:30', '1-2-1 Add Needs Mon 18:00', '1-2-1 Add Needs Mon 18:30', '1-2-1 Add Needs Sun 08:30', '11-15 Induction Mon 17:30', '11-15 Induction Sat 10.30', '16-18 Fri 19:30 Ag', '16:30 1-2-1 Inclusive Mon', '16:30 121 Inclusive Thurs', '17:00 1-2-1 Inclusive Mon', '17:00 121 Inclusive Thurs', '17:30 1-2-1 Inclusive Mon', '17:30 121 Inclusive Thurs', '18:00 1-2-1 Inclusive Mon', '18:00 121 Inclusive Thurs', '18:30 121 Inclusive Fri', '5', '50+ Supple Strength', '50+ Table Tennis', 'A & C 19-36 Fri 11:00 M.S', 'A & C 19-36 Thu 12:30 J.M', 'A & C 19-36 Thu 13:30 J.M', 'A & C 19-36 Tue 11:00 J.M', 'A & C 19-36 Wed 12:30 M.S', 'A & C 19-36m Sun 11:30 Rj', 'A & C 5-18 Fri 09:30 M.S', 'A & C 5-18 Fri 10:00 M.S', 'A & C 5-18 Fri 10:30 M.S', 'A & C 5-18 Thu 13:00 J.M', 'A & C 5-18 Tue 10:30 J.M', 'A & C 5-18m Sat 12:00 Sc', 'A & C 5-18m Sun 11:00 Rj', 'A & C 5-18m Wed 12:00 M.S', 'A Master Swi Thu 19:15 Ew', 'A& Ch 5-1

In [17]:
print(combined[(combined["venue_name"]=="Latchmere Leisure Centre")][["item_id","activity","facility_id" if "facility_id" in combined.columns else "activity","start_date"]].head(20))

                  item_id activity activity                 start_date
41   00000000000130062948     None     None  2026-06-24T16:30:00+00:00
108  00000000000130034777     None     None  2026-06-24T17:15:00+00:00
109  00000000000130034778     None     None  2026-07-01T17:15:00+00:00
153  00000000000130029556     None     None  2026-07-05T11:00:00+00:00
155  00000000000130062949     None     None  2026-07-01T16:30:00+00:00
158  00000000000130051790     None     None  2026-06-24T13:00:00+00:00
159  00000000000130051791     None     None  2026-07-01T13:00:00+00:00
164  00000000000130036917     None     None  2026-06-24T05:30:00+00:00
165  00000000000130036918     None     None  2026-07-01T05:30:00+00:00
207  00000000000130036401     None     None  2026-06-30T06:30:00+00:00
208  00000000000130036402     None     None  2026-07-07T06:30:00+00:00
214  00000000000130038173     None     None  2026-06-26T07:15:00+00:00
215  00000000000130038174     None     None  2026-07-03T07:15:00+00:00
216  0

In [18]:
print(venue_activity["venue_name"].str.strip().nunique(), "vs", venue_activity["venue_name"].nunique())
print(venue_activity["activity"].str.strip().nunique(), "vs", venue_activity["activity"].nunique())

50 vs 50
26 vs 26


In [19]:
print("City of London" in venue_activity["LAD24NM"].unique())

False


In [20]:
# Venues per borough — coverage check
print(venue_activity.groupby("LAD24NM")["venue_name"].nunique().sort_values(ascending=False))

# Distinct activities offered per borough — breadth of provision
print(venue_activity.groupby("LAD24NM")["activity"].nunique().sort_values(ascending=False))

# Provider presence per borough — is provision single-provider or mixed?
print(venue_activity.groupby("LAD24NM")["provider"].nunique())

# Most common activities citywide
print(venue_activity["activity"].value_counts().head(15))

# Total occurrence volume per borough — overall intensity of provision
print(venue_activity.groupby("LAD24NM")["n_occurrences"].sum().sort_values(ascending=False))

LAD24NM
Ealing                    10
Westminster                9
Wandsworth                 7
Havering                   6
Sutton                     4
Harrow                     3
Kingston upon Thames       3
Barking and Dagenham       2
Kensington and Chelsea     2
Brent                      2
Barnet                     1
Newham                     1
Name: venue_name, dtype: int64
LAD24NM
Westminster               17
Ealing                    12
Wandsworth                12
Brent                     11
Harrow                    10
Kensington and Chelsea     9
Sutton                     9
Havering                   9
Kingston upon Thames       8
Barking and Dagenham       7
Newham                     3
Barnet                     0
Name: activity, dtype: int64
LAD24NM
Barking and Dagenham      1
Barnet                    1
Brent                     1
Ealing                    1
Harrow                    1
Havering                  1
Kensington and Chelsea    1
Kingston upon Thames    

In [21]:
# Rows with at least one missing value (any column empty)
rows_with_any_na = combined[combined.isna().any(axis=1)]
print(f"{len(rows_with_any_na)} / {len(combined)} rows have at least one empty column")

# Break down WHICH columns are causing it
print(combined.isna().sum())

33519 / 42547 rows have at least one empty column
item_id           0
venue_name        0
lat               0
lon               0
activity      33042
start_date      477
provider          0
feed_type         0
LAD24NM           0
dtype: int64


In [22]:
rows_with_any_na_va = venue_activity[venue_activity.isna().any(axis=1)]
print(f"{len(rows_with_any_na_va)} / {len(venue_activity)} rows have at least one empty column")

print(venue_activity.isna().sum())

45 / 282 rows have at least one empty column
venue_name        0
activity         45
LAD24NM           0
provider          0
n_occurrences     0
lat               0
lon               0
dtype: int64
